# 03 — Double factorization with PyLIQTR and Qualtran

Two-electron integrals are decomposed as
$$h_{pqrs} = \sum_{\ell=1}^{L} g_\ell^{(1)} L_{pq}^{(\ell)} L_{rs}^{(\ell)},$$
where each factor matrix $L^{(\ell)}$ is itself diagonalized to give a second
factorization layer. Block-encoding cost drops from $\mathcal{O}(N^4)$ in the
plain LCU to $\mathcal{O}(N \Xi)$ with $\Xi$ the truncated rank.


In [19]:
import json
import math
import numpy as np
from openfermion import InteractionOperator, jordan_wigner

npz = np.load("data/integrals.npz")
labels = ["small", "medium", "target", "large", "xlarge", "xxlarge"]
all_data = {}
for label in labels:
    all_data[label] = {
        "nelec": int(npz[f"{label}_nelec"]),
        "norb": int(npz[f"{label}_norb"]),
        "e_core": float(npz[f"{label}_e_core"]),
        "h1e_eff": npz[f"{label}_h1e_eff"],
        "h2e_phys": npz[f"{label}_h2e_phys"],
    }
epsilon = 0.0016


Eigendecompose the symmetrized two-electron matrix; sum eigenvalue magnitudes
for the 2e contribution to lambda; combine with $\lambda_{1e} = \sum_{pq}|h_{pq}^{\text{eff}}|$.

In [20]:
def double_factorize(h1e, h2e, norb, threshold=1e-6):
    h2e_flat = h2e.reshape(norb**2, norb**2)
    h2e_sym = 0.5 * (h2e_flat + h2e_flat.T)
    eigvals, eigvecs = np.linalg.eigh(h2e_sym)
    mask = np.abs(eigvals) > threshold
    eigvals = eigvals[mask]
    eigvecs = eigvecs[:, mask]
    rank = len(eigvals)
    return {
        "lambda_1e": float(np.sum(np.abs(h1e))),
        "lambda_2e": float(np.sum(np.abs(eigvals))),
        "rank": int(rank),
        "eigvals": eigvals,
    }


df = {}
for label in labels:
    d = all_data[label]
    norb = d["norb"]
    df = double_factorize(d["h1e_eff"], d["h2e_phys"], norb)

    lam_df = df["lambda_1e"] + df["lambda_2e"]
    rank = df["rank"]
    log2_rank = max(1, math.ceil(math.log2(rank)))

    # Lee/Babbush-style walk cost
    toff_per_walk = 2 * (norb * rank + rank * log2_rank + 2)
    t_per_walk = 4 * toff_per_walk

    prec = int(np.ceil(np.log2(lam_df / epsilon))) + 1
    qpe_rounds = 2 ** prec
    t_total = qpe_rounds * t_per_walk

    ancilla = log2_rank + math.ceil(math.log2(norb)) + prec
    logical_qubits = 2 * norb + ancilla + 1

    df[label] = {
        "active_space": f"({d['nelec']}e, {norb}o)",
        "lambda_1e": df["lambda_1e"],
        "lambda_2e": df["lambda_2e"],
        "lambda_total": float(lam_df),
        "df_rank": rank,
        "qpe_bits": int(prec),
        "qpe_rounds": int(qpe_rounds),
        "t_per_walk": int(t_per_walk),
        "t_total": int(t_total),
        "logical_qubits": int(logical_qubits),
    }

    print(f"({d['nelec']}e,{norb}o): rank={rank:>4}  lambda={lam_df:>7.2f}  "
          f"QPE={prec:>2}  T={t_total:>16,}  Q_L={logical_qubits}")

(4e,4o): rank=  16  lambda=   7.86  QPE=14  T=      17,039,360  Q_L=29
(8e,8o): rank=  64  lambda=  34.85  QPE=16  T=     470,810,624  Q_L=42
(12e,12o): rank= 144  lambda=  79.62  QPE=17  T=   3,021,996,032  Q_L=54
(16e,16o): rank= 256  lambda= 142.34  QPE=18  T=  12,889,096,192  Q_L=63
(20e,20o): rank= 400  lambda= 229.15  QPE=19  T=  48,662,315,008  Q_L=74
(24e,24o): rank= 576  lambda= 339.60  QPE=19  T=  82,149,638,144  Q_L=83


## Save DF results

In [21]:
out = {
    "df": df,
}
with open("data/df_results.json", "w") as f:
    json.dump(out, f, indent=2, default=str)
print("saved data/df_results.json")

saved data/df_results.json
